# Lab 6: Feature Engineering, Evaluation und Tuning

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Datenordner finden: Notebook liegt in labs/ oder loesungen/, die Daten in data/
DATA = next(p for p in [Path("data"), Path("../data"), Path("../../data")] if p.exists())
print("Datenordner:", DATA)

Dieses Lab gehört zu **Teil 6: Modelltraining, Feature Engineering, Hyperparameter und Evaluation**.

## Lernziele

- Neue Merkmale bilden, schiefe Größen logarithmieren und Kategorien mit One-Hot kodieren
- Overfitting am Abstand zwischen Trainingsfehler und Fehler auf neuen Daten erkennen
- Accuracy, Precision, Recall, F1 sowie MAE, MSE, RMSE und R² von Hand rechnen und mit scikit-learn prüfen
- Modelle mit Cross-Validation prüfen und Hyperparameter mit `GridSearchCV` suchen
- Bei ungleichen Klassen die passende Kennzahl wählen, Klassen gewichten und die ROC-Kurve lesen

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                     StratifiedKFold, GridSearchCV, RandomizedSearchCV)
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, roc_auc_score,
                             mean_absolute_error, mean_squared_error, r2_score)

titanic = pd.read_csv(DATA / "titanic.csv")
print(titanic.shape)

## Block 1: Feature Engineering auf Titanic

1. Bilden Sie `FamilySize` (Geschwister/Partner + Eltern/Kinder + die Person selbst) und `IsAlone` (1, wenn `FamilySize` gleich 1). Vergleichen Sie die Überlebensrate. Erwartet: 537 Alleinreisende, größte Familie 11 Personen. Überlebensrate allein 0.304, mit Familie 0.506.
2. Bilden Sie `Altersgruppe` mit `pd.cut` (Grenzen 0, 12, 18, 30, 50, 80) und geben Sie die Überlebensrate je Gruppe aus. Erwartet: Kind 0.58, Jugendlich 0.43, Jung 0.36, Erwachsen 0.42, Senior 0.34. 177 Personen bleiben ohne Gruppe, weil `Age` fehlt.
3. Logarithmieren Sie `Fare` mit `np.log1p` und zeichnen Sie die Histogramme vorher und nachher. Erwartet: Schiefe (`skew`) sinkt von 4.79 auf 0.39.
4. Kodieren Sie `Sex` und `Embarked` mit `pd.get_dummies(..., drop_first=True)`. Kodieren Sie danach `Sex` und `Pclass` mit `OneHotEncoder` und lassen Sie beide Wege eine neue Zeile mit der unbekannten Kategorie `"unbekannt"` verarbeiten. Erwartet: neue Spalten `Sex_male`, `Embarked_Q`, `Embarked_S`. Der `OneHotEncoder` liefert `[[0. 0. 0. 0. 1.]]`, `get_dummies` erfindet eine Spalte `Sex_unbekannt`, die das Modell nicht kennt.

In [ ]:
# Aufgabe 1: FamilySize und IsAlone
# Tipp: Spalten addieren; (Bedingung).astype(int) macht aus True/False 1/0
titanic["FamilySize"] = ...
titanic["IsAlone"] = ...
# print(titanic["IsAlone"].sum(), titanic["FamilySize"].max())
# print(titanic.groupby("IsAlone")["Survived"].mean().round(3))

In [ ]:
# Aufgabe 2: Altersgruppe (Binning)
# Tipp: pd.cut(spalte, bins=[...], labels=[...]); beim groupby observed=True angeben
gruppen = ["Kind", "Jugendlich", "Jung", "Erwachsen", "Senior"]
titanic["Altersgruppe"] = ...
# print(titanic.groupby("Altersgruppe", observed=True)["Survived"].mean().round(2))
# print(titanic["Altersgruppe"].isna().sum())

In [ ]:
# Aufgabe 3: Fare logarithmieren
# Tipp: np.log1p rechnet log(1 + x) und verträgt auch Fare = 0
titanic["Fare_log"] = ...
# print(titanic["Fare"].skew().round(2), titanic["Fare_log"].skew().round(2))
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))
# ax1.hist(titanic["Fare"], bins=40)
# ax2.hist(titanic["Fare_log"], bins=40)

In [ ]:
# Aufgabe 4: One-Hot mit pandas gegen OneHotEncoder
# Tipp: OneHotEncoder(handle_unknown="ignore", sparse_output=False), dann fit_transform und get_feature_names_out
kodiert = ...     # pd.get_dummies(titanic, columns=[...], drop_first=True)
# print([c for c in kodiert.columns if c.startswith(("Sex_", "Embarked_"))])

ohe = ...
# X_kat = ohe.fit_transform(titanic[["Sex", "Pclass"]])
# print(ohe.get_feature_names_out())

neu = pd.DataFrame({"Sex": ["unbekannt"], "Pclass": [3]})     # Kategorie, die im Training nie vorkam
# print(ohe.transform(neu))
# print(pd.get_dummies(neu, columns=["Sex", "Pclass"]).columns.tolist())

## Block 2: Overfitting am Polynombeispiel

Zwölf Messpunkte einer Sinuskurve mit Rauschen. Ein Polynom vom Grad 1 ist eine Gerade, Grad 11 hat 12 Koeffizienten für 12 Punkte.

1. Übernehmen Sie das Beispiel von der Folie: Polynome vom Grad 1, 4 und 11 anpassen, Fehler (MSE) auf den Trainingspunkten und auf 50 neuen Punkten ausgeben. Erwartet: Training 0.222, 0.020, 0.000. Neu 0.172, 0.010, 0.111.
2. Tabellieren Sie beide Fehler für alle Grade von 1 bis 11 und zeichnen Sie sie über dem Grad. Bei welchem Grad ist der Fehler auf neuen Daten am kleinsten? Erwartet: Grad 5 mit 0.004. Der Trainingsfehler sinkt dagegen immer weiter.

In [ ]:
# Daten für Block 2
rng = np.random.default_rng(42)
X_p = np.linspace(-3, 3, 12)                         # 12 Messpunkte
y_p = np.sin(X_p) + rng.normal(0, 0.15, 12)          # Sinus plus Rauschen
X_neu = np.linspace(-2.9, 2.9, 50)                   # neue Punkte
y_neu = np.sin(X_neu)

In [ ]:
# Aufgabe 1: Grad 1, 4 und 11
# Tipp: p = Polynomial.fit(X_p, y_p, grad); p(X_p) liefert die Vorhersagen
for grad in [1, 4, 11]:
    p = ...
    # mse_train = np.mean((p(X_p) - y_p) ** 2)
    # mse_neu = ...
    # print(f"Grad {grad:2d}: Training {mse_train:.3f}, neu {mse_neu:.3f}")

In [ ]:
# Aufgabe 2: Alle Grade von 1 bis 11
# Tipp: Liste von Dictionaries, pd.DataFrame(...).set_index("Grad"); idxmin() findet den besten Grad
zeilen = []
for grad in range(1, 12):
    ...
fehler = pd.DataFrame(zeilen)
fehler

## Block 3: Kennzahlen von Hand

Ein Test auf eine Krankheit wurde an 100 Personen geprüft (Klasse 1 = krank). Die Konfusionsmatrix, Zeilen = Wahrheit, Spalten = Vorhersage:

| | vorhergesagt 0 | vorhergesagt 1 |
|---|---|---|
| **wirklich 0** | TN = 50 | FP = 10 |
| **wirklich 1** | FN = 5 | TP = 35 |

1. Rechnen Sie Accuracy, Precision, Recall und F1 mit den vier Zahlen aus. Erwartet: 0.85, 0.778, 0.875, 0.824.
2. Prüfen Sie Ihre Werte mit `sklearn.metrics`. Die passenden Arrays `y_true` und `y_pred` sind vorgegeben. Erwartet: `confusion_matrix` liefert `[[50 10] [5 35]]`, die vier Kennzahlen stimmen mit Aufgabe 1 überein.
3. Regression: Rechnen Sie MAE, MSE, RMSE und R² für fünf Wertepaare in NumPy und prüfen Sie mit scikit-learn. Erwartet: MAE 1.2, MSE 2.4, RMSE 1.549, R² 0.774.

In [ ]:
# Aufgabe 1: Vier Kennzahlen aus vier Zahlen
# Tipp: Precision schaut auf die Spalte "vorhergesagt 1", Recall auf die Zeile "wirklich 1"
tn, fp, fn, tp = 50, 10, 5, 35
accuracy = ...
precision = ...
recall = ...
f1 = ...
print(accuracy, precision, recall, f1)

In [ ]:
# Aufgabe 2: Mit scikit-learn prüfen
# np.repeat baut aus den vier Zahlen die 100 Einzelfälle: 50 mal (0, 0), 10 mal (0, 1), 5 mal (1, 0), 35 mal (1, 1)
y_true = np.repeat([0, 0, 1, 1], [50, 10, 5, 35])
y_pred = np.repeat([0, 1, 0, 1], [50, 10, 5, 35])
# Tipp: Reihenfolge der Argumente: erst die Wahrheit, dann die Vorhersage
# print(confusion_matrix(..., ...))
# print(accuracy_score(..., ...), precision_score(..., ...), ...)

In [ ]:
# Aufgabe 3: Regressionskennzahlen in NumPy
# Tipp: fehler = y_w - y_v; np.abs, ** 2, np.mean, np.sqrt; R² = 1 - Summe(fehler²) / Summe((y_w - Mittelwert)²)
y_w = np.array([3, 5, 8, 10, 12])      # wahre Werte
y_v = np.array([2, 5, 11, 9, 13])      # Vorhersagen
fehler = y_w - y_v
mae = ...
mse = ...
rmse = ...
r2 = ...
print(mae, mse, rmse, r2)
# print(mean_absolute_error(y_w, y_v), mean_squared_error(y_w, y_v), r2_score(y_w, y_v))

## Block 4: Cross-Validation

Ein einzelner Split hängt vom Zufall ab. Cross-Validation lässt jeden von fünf Blöcken einmal Prüfdaten sein. Datensatz: Brustkrebs (0 = bösartig, 1 = gutartig).

1. Prüfen Sie `DecisionTreeClassifier(max_depth=3, random_state=1)` mit `cross_val_score` und dem vorgegebenen `StratifiedKFold`. Geben Sie die fünf Scores, Mittel und Streuung aus. Erwartet: `[0.895 0.947 0.921 0.939 0.885]`, Mittel 0.917, Streuung 0.024.
2. Wiederholen Sie das mit `RandomForestClassifier(random_state=1)`. Erwartet: Mittel 0.954, Streuung 0.015.
3. Nehmen Sie für den Baum statt `StratifiedKFold` ein `KFold(n_splits=5)` ohne Mischen und geben Sie je Fold den Anteil der Klasse 1 in den Prüfdaten aus. Erwartet: Anteile 0.40, 0.57, 0.65, 0.75, 0.77 (stratifiziert: immer 0.62 oder 0.63). Accuracy im Mittel 0.909, Streuung 0.032.

In [ ]:
# Daten für Block 4 und 5
X_bc, y_bc = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
print(X_bc.shape, np.bincount(y_bc))

In [ ]:
# Aufgabe 1: Fünf Scores statt einem
# Tipp: cross_val_score(modell, X_bc, y_bc, cv=cv, scoring="accuracy")
baum = DecisionTreeClassifier(max_depth=3, random_state=1)
scores = ...
# print(scores.round(3))
# print(f"Mittel: {scores.mean():.3f}, Streuung: {scores.std():.3f}")

In [ ]:
# Aufgabe 2: Random Forest mit derselben Aufteilung
wald = RandomForestClassifier(random_state=1)
scores_wald = ...
# ...

In [ ]:
# Aufgabe 3: KFold ohne Mischen gegen StratifiedKFold
# Tipp: for train_idx, test_idx in kf.split(X_bc, y_bc): y_bc[test_idx].mean() ist der Anteil der Klasse 1
kf = KFold(n_splits=5)
# for train_idx, test_idx in kf.split(X_bc, y_bc):
#     print(...)
# scores_kf = cross_val_score(baum, X_bc, y_bc, cv=kf)

## Block 5: GridSearchCV

`GridSearchCV` probiert jede Kombination eines Gitters per Cross-Validation und trainiert danach das beste Modell auf allen Trainingsdaten neu. Achten Sie auf die Laufzeit: Kandidaten mal Folds ergibt die Zahl der Trainingsläufe.

1. Suchen Sie die beste Tiefe für einen `DecisionTreeRegressor(random_state=1)` auf California Housing: `max_depth` in `[2, 4, 6, 8, 10, 12, 16, 20]`, `cv=5`, `scoring="r2"`. Geben Sie `cv_results_` als Tabelle aus und prüfen Sie das beste Modell auf dem Testteil. Erwartet: beste Tiefe 8, R² in der Cross-Validation 0.682, auf dem Testteil 0.710. Tiefe 2 kommt nur auf 0.441 (Underfitting), Tiefe 20 fällt auf 0.600 zurück (Overfitting). 40 Trainingsläufe.
2. Suchen Sie für einen Random Forest auf dem Brustkrebs-Datensatz `max_depth` in `[2, 3, 5, None]` und `n_estimators` in `[50, 100]`, mit `scoring="f1"` und dem `cv` aus Block 4. Erwartet: 8 Kombinationen mal 5 Folds = 40 Trainingsläufe, beste Kombination `max_depth=None`, `n_estimators=100`, F1 in der Cross-Validation 0.965, auf dem Testteil 0.966.

In [ ]:
# Daten für Block 5, Aufgabe 1
haeuser = pd.read_csv(DATA / "california_housing.csv")
X_h, y_h = haeuser.drop(columns="MedHouseVal"), haeuser["MedHouseVal"]
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(X_h, y_h, test_size=0.2, random_state=1)

# Daten für Block 5, Aufgabe 2 (stratify hält das Klassenverhältnis in beiden Teilen gleich)
X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=1)

In [ ]:
# Aufgabe 1: Beste Tiefe für einen Regressionsbaum
# Tipp: GridSearchCV(DecisionTreeRegressor(random_state=1), {"max_depth": tiefen}, cv=5, scoring="r2")
tiefen = [2, 4, 6, 8, 10, 12, 16, 20]
suche = ...
# suche.fit(X_h_train, y_h_train)
# print(suche.best_params_, suche.best_score_)
# print(suche.score(X_h_test, y_h_test))
# pd.DataFrame(suche.cv_results_)[["param_max_depth", "mean_test_score", "rank_test_score"]]

In [ ]:
# Aufgabe 2: Tiefe und Baumzahl eines Random Forest
# Tipp: kleines Gitter halten, jeder weitere Wert vervielfacht die Laufzeit
gitter = {"max_depth": [2, 3, 5, None], "n_estimators": [50, 100]}
suche_rf = ...
# suche_rf.fit(X_bc_train, y_bc_train)
# print(suche_rf.best_params_, suche_rf.best_score_)
# print(f1_score(y_bc_test, suche_rf.predict(X_bc_test)))

## Block 6: Ungleiche Klassen

2000 künstliche Fälle, davon rund 5 Prozent in der Klasse 1 (die seltene, interessante Klasse).

1. Trainieren Sie ein „Modell", das immer die Mehrheit vorhersagt: `DummyClassifier(strategy="most_frequent")`. Geben Sie Accuracy, Recall und Konfusionsmatrix aus. Erwartet: 22 Fälle der Klasse 1 im Testteil, Accuracy 0.945, Recall 0.0.
2. Trainieren Sie `LogisticRegression(max_iter=1000)` einmal ohne und einmal mit `class_weight="balanced"`. Tabellieren Sie Accuracy, Precision und Recall. Erwartet: ohne Gewichte 0.983, 0.941, 0.727. Mit Gewichten 0.915, 0.389, 0.955.
3. Verschieben Sie beim Modell ohne Gewichte die Schwelle auf 0.3 und 0.7. Erwartet: Schwelle 0.3 ergibt Precision 0.944 und Recall 0.773. Schwelle 0.7 ergibt 0.923 und 0.545.
4. Zeichnen Sie die ROC-Kurven beider Modelle in ein Diagramm und berechnen Sie die AUC. Erwartet: AUC 0.977 ohne und 0.971 mit Gewichten. Das Dummy-Modell hat AUC 0.5.

In [ ]:
# Daten für Block 6
X_u, y_u = make_classification(n_samples=2000, weights=[0.95, 0.05], random_state=1)
X_u_train, X_u_test, y_u_train, y_u_test = train_test_split(
    X_u, y_u, test_size=0.2, stratify=y_u, random_state=1)
print(np.bincount(y_u), np.bincount(y_u_test))

In [ ]:
# Aufgabe 1: Immer die Mehrheit vorhersagen
# Tipp: DummyClassifier hat dieselbe Schnittstelle wie jedes Modell: fit, predict
dummy = ...
# dummy.fit(X_u_train, y_u_train)
# y_dummy = dummy.predict(X_u_test)
# print(accuracy_score(y_u_test, y_dummy), recall_score(y_u_test, y_dummy))
# print(confusion_matrix(y_u_test, y_dummy))

In [ ]:
# Aufgabe 2: Ohne und mit class_weight="balanced"
# Tipp: zwei Modelle in einem Dictionary, Schleife, Liste von Dictionaries
kandidaten = {
    "ohne Gewichte": LogisticRegression(max_iter=1000),
    # zweites Modell ergänzen
}
zeilen = []
for name, modell in kandidaten.items():
    ...
pd.DataFrame(zeilen)

In [ ]:
# Aufgabe 3: Schwelle verschieben
# Tipp: proba = modell.predict_proba(X_u_test)[:, 1]; (proba >= schwelle).astype(int)
ohne = kandidaten["ohne Gewichte"]
for schwelle in [0.3, 0.5, 0.7]:
    ...

In [ ]:
# Aufgabe 4: ROC-Kurve und AUC
# Tipp: fpr, tpr, _ = roc_curve(y_u_test, proba); roc_auc_score(y_u_test, proba)
fig, ax = plt.subplots(figsize=(5, 5))
for name, modell in kandidaten.items():
    ...
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")     # Diagonale: Raten
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
plt.show()

## Zusatzaufgaben

1. Bauen Sie in die Vorhersagen aus Block 3 einen groben Ausreißer ein (letzter Wert 30 statt 13). Welche Kennzahl reagiert am stärksten? Erwartet: MAE 4.6, RMSE 8.185, R² -5.297. Ein negatives R² heißt: schlechter als immer den Mittelwert vorherzusagen.
2. Prüfen Sie in `versicherte.csv`, ob `leistungsausgaben_eur` ein Kandidat für die Log-Transformation ist: Schiefe vorher und nach `np.log1p`. Erwartet: Schiefe 6.07 vorher, 0.68 nachher.
3. Wiederholen Sie die Suche aus Block 5, Aufgabe 2, mit `RandomizedSearchCV` und `n_iter=5`. Erwartet: 25 statt 40 Trainingsläufe, F1 in der Cross-Validation wieder 0.965.

In [ ]:
# Zusatz 1: Ein Ausreißer in den Vorhersagen
y_a = y_v.copy()
y_a[4] = 30
# ...

In [ ]:
# Zusatz 2: Leistungsausgaben logarithmieren?
versicherte = pd.read_csv(DATA / "versicherte.csv")
# ...

In [ ]:
# Zusatz 3: Zufallssuche statt Gittersuche
# Tipp: RandomizedSearchCV(modell, param_distributions=gitter, n_iter=5, cv=cv, scoring="f1", random_state=1)
zufall = ...
# ...

## Was Sie mitnehmen

- Gute Merkmale entstehen aus Fachwissen: kombinieren, gruppieren, logarithmieren, kodieren. Ein `OneHotEncoder` merkt sich seine Kategorien, `pd.get_dummies` nicht.
- Der Trainingsfehler allein sagt nichts. Entscheidend ist der Abstand zu neuen Daten, und Cross-Validation zeigt neben dem Mittel auch die Streuung. Hyperparameter sucht `GridSearchCV`, der Testteil kommt genau einmal am Ende zum Einsatz.
- Lesen Sie die Konfusionsmatrix vor der Accuracy. Bei ungleichen Klassen zählen Recall, Precision, F1 und AUC. Klassengewichte und Schwelle sind fachliche Entscheidungen über Fehlerkosten.